# Cassava — Data Integration and Statistical Analysis

This notebook processes LC-MS/MS metabolomics data for cassava (*Manihot esculenta*) samples from batch `b3_cassavaonly`. It consolidates MS2 annotations from multiple sources into a single feature table, runs differential abundance statistics between control and fermented conditions, and prepares compound structures for downstream metabolic transformation prediction.

- **Section 1** — MS2 annotation processing (SIRIUS + GNPS + Suspect)
- **Section 2** — Quantitative analysis (normalization, imputation, FDR-corrected statistics)
- **Section 3** — BioTransformer prep (InChIKey → SMILES)

**Inputs:** SIRIUS/CANOPUS structure and classification predictions, GNPS spectral library and suspect list matches, MZmine quantitative feature table.

**Outputs:** `b3_cassavaonly_ms2_annotations.csv`, `b3_cassavaonly_feature_list_imputed_log2.csv`, `b3_cassavaonly_fdr_cassava_vs_fermented_cassava.csv`, BioTransformer input file.

In [1]:
import pandas as pd
import numpy as np
import re
import os
from scipy import stats
from statsmodels.stats.multitest import multipletests

BATCH_ID    = "b3_cassavaonly"
DATA_PATH   = f"../data/processed_data/{BATCH_ID}_biot"  # New biotransformer SIRIUS results
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Batch: {BATCH_ID}")
print(f"Data path: {DATA_PATH}")


Batch: b3_cassavaonly
Data path: ../data/processed_data/b3_cassavaonly_biot


---
## Section 1 — MS2 Annotation Processing

Annotations from three sources are consolidated per feature: SIRIUS structure predictions (filtered at confidence ≥ 0.64), CANOPUS chemical classifications (NPC and ClassyFire taxonomies), and GNPS spectral library matches. A GNPS suspect list provides additional tentative identifications. Where both GNPS and SIRIUS annotate the same feature, InChIKey agreement is checked.

### 1.1 Load SIRIUS Structure Identifications

In [2]:
sirius_structure_annotations = pd.read_csv(
    f"{DATA_PATH}/structure_identifications.tsv", sep="\t"
)
print(f"SIRIUS structure annotations shape: {sirius_structure_annotations.shape}")
sirius_structure_annotations.head()


SIRIUS structure annotations shape: (694, 27)


,structurePerIdRank,formulaRank,ConfidenceScoreExact,ConfidenceScoreApproximate,CSI:FingerIDScore,ZodiacScore,SiriusScoreNormalized,SiriusScore,molecularFormula,adduct,...,links,dbflags,ionMass,retentionTimeInSeconds,retentionTimeInMinutes,formulaId,alignedFeatureId,compoundId,mappingFeatureId,overallFeatureQuality
0,1,3,0.017,0.017,-404.776,NaN,0.076,17.814,C17H28N6O6,[M + H]+,...,CHEBI:(162963);PUBCHEM:(17907398 145455395),34,413.211,36,0.607,816974037128036617,816973399728025968,816973399723831663,8,NaN
1,1,1,0.114,0.114,-81.704,NaN,0.996,51.965,C6H11O5P,[M + K]+,...,NORMAN:(NS00033255);PUBCHEMANNOTATIONSAFETYAND...,137506324482,232.998,37,0.611,816974022372471171,816973399870632309,816973399866438004,13,NaN
2,1,1,0.036,0.036,-128.477,NaN,0.997,21.443,C12H6O7,[M + K]+,...,PUBCHEM:(153523186),2,300.973,37,0.612,816974031184706091,816973399983878522,816973399983878521,17,NaN
3,1,1,0.457,0.457,-62.494,NaN,0.993,59.232,C7H13O5P,[M + K]+,...,PUBCHEM:(5866391 3580242),2,247.014,37,0.618,816974067574497519,816973400092930431,816973400088736126,48,NaN
4,1,10,-inf,-inf,-177.990,NaN,0.004,-0.516,C9H13Cl3N2O2S3,[M + H]+,...,PUBCHEM:(54256095 24837726),2,382.927,37,0.618,816974094527097454,816973400201982340,816973400201982339,49,NaN


In [3]:
confidence_threshold = 0.64
sirius_best = sirius_structure_annotations[
    sirius_structure_annotations['structurePerIdRank'] == 1
].copy()
sirius_best['high_confidence_sirius'] = (
    sirius_best['ConfidenceScoreApproximate'] >= confidence_threshold
)
sirius_cols_keep = [
    'mappingFeatureId', 'ConfidenceScoreApproximate',
    'InChIkey2D', 'InChI', 'name', 'smiles', 'dbflags', 'links',
    'high_confidence_sirius'
]
sirius_cols_keep = [c for c in sirius_cols_keep if c in sirius_best.columns]
sirius_struct_clean = sirius_best[sirius_cols_keep].copy()
print(f"SIRIUS rank-1 features: {len(sirius_struct_clean)}")
print(f"High-confidence (>={confidence_threshold}): {sirius_struct_clean['high_confidence_sirius'].sum()}")


SIRIUS rank-1 features: 694
High-confidence (>=0.64): 93


### 1.2 Load CANOPUS Annotations

In [4]:
canopus_annotations = pd.read_csv(
    f"{DATA_PATH}/canopus_structure_summary.tsv", sep="\t"
)
print(f"CANOPUS shape: {canopus_annotations.shape}")
canopus_annotations.head()


CANOPUS shape: (694, 29)


,formulaRank,molecularFormula,adduct,precursorFormula,NPC#pathway,NPC#pathway Probability,NPC#superclass,NPC#superclass Probability,NPC#class,NPC#class Probability,...,ClassyFire#most specific class Probability,ClassyFire#all classifications,ionMass,retentionTimeInSeconds,retentionTimeInMinutes,formulaId,alignedFeatureId,compoundId,mappingFeatureId,overallFeatureQuality
0,3,C17H28N6O6,[M + H]+,C17H29N6O6+,Amino acids and Peptides,0.719,Small peptides,0.669,Dipeptides,0.248,...,0.589,"Organic compounds; Amino acids, peptides, and ...",413.211,36,0.607,816974037128036617,816973399728025968,816973399723831663,8,NaN
1,1,C6H11O5P,[M + K]+,C6H11KO5P+,Fatty acids,0.822,Fatty Acids and Conjugates,0.690,Halogenated fatty acids,0.026,...,0.620,Organic compounds; Lipids and lipid-like molec...,232.998,37,0.611,816974022372471171,816973399870632309,816973399866438004,13,NaN
2,1,C12H6O7,[M + K]+,C12H6KO7+,Shikimates and Phenylpropanoids,0.614,Naphthalenes,0.461,Naphthoquinones,0.535,...,0.508,Organic compounds; Organoheterocyclic compound...,300.973,37,0.612,816974031184706091,816973399983878522,816973399983878521,17,NaN
3,1,C7H13O5P,[M + K]+,C7H13KO5P+,Fatty acids,0.911,Fatty Acids and Conjugates,0.720,Halogenated fatty acids,0.014,...,0.548,Organic compounds; Lipids and lipid-like molec...,247.014,37,0.618,816974067574497519,816973400092930431,816973400088736126,48,NaN
4,10,C9H13Cl3N2O2S3,[M + H]+,C9H14Cl3N2O2S3+,Fatty acids,0.230,Oligopeptides,0.383,Primary amides,0.056,...,0.536,Organic compounds; Organoheterocyclic compound...,382.927,37,0.618,816974094527097454,816973400201982340,816973400201982339,49,NaN


In [5]:
canopus_cols_desired = [
    'mappingFeatureId', 'molecularFormula', 'adduct',
    'NPC#pathway', 'NPC#pathway Probability',
    'NPC#superclass', 'NPC#superclass Probability',
    'NPC#class', 'NPC#class Probability',
    'ClassyFire#superclass', 'ClassyFire#superclass probability',
    'ClassyFire#class', 'ClassyFire#class Probability',
    'ClassyFire#subclass', 'ClassyFire#subclass Probability',
    'ClassyFire#level 5', 'ClassyFire#level 5 Probability',
    'ClassyFire#most specific class', 'ClassyFire#most specific class Probability'
]
sirius_canopus = canopus_annotations[
    [c for c in canopus_cols_desired if c in canopus_annotations.columns]
].copy()
print(f"CANOPUS selected shape: {sirius_canopus.shape}")


CANOPUS selected shape: (694, 19)


### 1.3 Merge SIRIUS Structure + CANOPUS

In [6]:
sirius_annotations = pd.merge(sirius_canopus, sirius_struct_clean, on='mappingFeatureId', how='left')
sirius_annotations = sirius_annotations.rename(
    columns={col: f"sirius_{col}" for col in sirius_annotations.columns if col != 'mappingFeatureId'}
)
print(f"SIRIUS merged shape: {sirius_annotations.shape}")
print(f"Features with structure: {sirius_annotations['sirius_InChIkey2D'].notna().sum()}")
print(f"High-confidence: {(sirius_annotations['sirius_high_confidence_sirius'] == True).sum()}")


SIRIUS merged shape: (694, 27)
Features with structure: 694
High-confidence: 93


### 1.4 Load GNPS Annotations

In [7]:
gnps_raw = pd.read_csv(f"{DATA_PATH}/merged_results_with_gnps.tsv", sep="\t")
gnps_cols_desired = [
    '#Scan#', 'Compound_Name', 'Smiles', 'INCHI', 'InChIKey-Planar',
    'superclass', 'class', 'subclass',
    'npclassifier_superclass', 'npclassifier_class', 'npclassifier_pathway'
]
gnps_annotations = gnps_raw[
    [c for c in gnps_cols_desired if c in gnps_raw.columns]
].rename(columns={
    '#Scan#': 'mappingFeatureId',
    'Compound_Name': 'gnps_Compound_Name',
    'Smiles': 'gnps_smiles',
    'INCHI': 'gnps_InChI',
    'InChIKey-Planar': 'gnps_InChIkey2D',
    'superclass': 'gnps_ClassyFire#superclass',
    'class': 'gnps_ClassyFire#class',
    'subclass': 'gnps_ClassyFire#subclass',
    'npclassifier_superclass': 'gnps_NPC#superclass',
    'npclassifier_class': 'gnps_NPC#class',
    'npclassifier_pathway': 'gnps_NPC#pathway'
})
print(f"GNPS cleaned shape: {gnps_annotations.shape}")
gnps_annotations.head()


GNPS cleaned shape: (36, 11)


,mappingFeatureId,gnps_Compound_Name,gnps_smiles,gnps_InChI,gnps_InChIkey2D,gnps_ClassyFire#superclass,gnps_ClassyFire#class,gnps_ClassyFire#subclass,gnps_NPC#superclass,gnps_NPC#class,gnps_NPC#pathway
0,3307,GalCer(d18:2/18:1); [M+H]+ C42H78N1O8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2741,PC(20:5/0:0); [M+H]+ C28H49N1O7P1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3053,PC(0:0/18:1); [M+H]+ C26H53N1O7P1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2323,Spectral Match to 1-Palmitoyl-2-hydroxy-sn-gly...,CCCCCCCCCCCCCCCC(=O)OC[C@H](COP(=O)(O)OCCN)O,InChI=1S/C21H44NO7P/c1-2-3-4-5-6-7-8-9-10-11-1...,YVYMBNSKXOXSKW,Lipids and lipid-like molecules,Glycerophospholipids,Glycerophosphoethanolamines,Glycerophospholipids,Glycerophosphoethanolamines,Fatty acids
4,2915,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 1.5 Load GNPS Suspect Annotations

In [8]:
gnps_suspect_raw = pd.read_csv(
    f"{DATA_PATH}/merged_results_with_gnps_suspect.tsv", sep="\t"
)
gnps_suspect_raw = gnps_suspect_raw[
    gnps_suspect_raw["LibraryName"] == "GNPS-SUSPECTLIST.mgf"
]
gnps_library_ids = gnps_annotations["mappingFeatureId"].tolist()
gnps_suspect_raw = gnps_suspect_raw[
    ~gnps_suspect_raw["#Scan#"].isin(gnps_library_ids)
]
gnps_suspect_annotations = (
    gnps_suspect_raw[['#Scan#', 'Compound_Name']]
    .rename(columns={'#Scan#': 'mappingFeatureId', 'Compound_Name': 'gnps_name'})
)
print(f"GNPS suspect (after filtering): {gnps_suspect_annotations.shape}")
gnps_suspect_annotations.head()


GNPS suspect (after filtering): (53, 2)


,mappingFeatureId,gnps_name
14,3051,NaN
18,2943,NaN
19,3021,NaN
24,3433,Suspect related to N-Palmitoyl-D-sphingosine (...
30,2747,Suspect related to Spectral Match to 1-(9Z-Oct...


In [9]:
def parse_suspect_annotation(text):
    """Parse GNPS suspect annotation string into structured fields."""
    if pd.isna(text) or not isinstance(text, str):
        return {'compound_name': None, 'sirius_formula': None, 'buddy_formula': None,
                'delta_mz': None, 'explanation': None, 'adduct': None}
    result = {}
    m = re.search(r'Suspect related to\s+(?:Spectral Match to\s+)?(.+?)\s+(?:from [A-Z0-9]+\s+)?\(predicted', text)
    result['compound_name'] = m.group(1).strip() if m else None
    for key, pattern in [
        ('sirius_formula', r'SIRIUS:\s*([A-Z][A-Za-z0-9]+)'),
        ('buddy_formula',  r'BUDDY:\s*([A-Z][A-Za-z0-9]+)'),
        ('explanation',    r'putative explanation:\s*([^;]+)'),
        ('adduct',         r'\[([^\]]+)\](?!.*\[)'),
    ]:
        m2 = re.search(pattern, text)
        result[key] = m2.group(1).strip() if m2 else None
    m3 = re.search(r'delta m/z\s+([\-\d.]+)', text)
    result['delta_mz'] = float(m3.group(1)) if m3 else None
    return result

parsed_df = pd.DataFrame(
    gnps_suspect_annotations['gnps_name'].apply(parse_suspect_annotation).tolist(),
    index=gnps_suspect_annotations.index
)
suspect_final = pd.concat([gnps_suspect_annotations, parsed_df], axis=1).rename(columns={
    'compound_name': 'gnps_suspect_compound_name',
    'sirius_formula': 'gnps_suspect_sirius_formula',
    'buddy_formula':  'gnps_suspect_buddy_formula',
    'delta_mz':       'gnps_suspect_delta_mz',
    'explanation':    'gnps_suspect_explanation',
    'adduct':         'gnps_suspect_adduct',
})
print(f"Suspect final shape: {suspect_final.shape}")
suspect_final.head()


Suspect final shape: (53, 8)


,mappingFeatureId,gnps_name,gnps_suspect_compound_name,gnps_suspect_sirius_formula,gnps_suspect_buddy_formula,gnps_suspect_delta_mz,gnps_suspect_explanation,gnps_suspect_adduct
14,3051,NaN,None,None,None,NaN,None,None
18,2943,NaN,None,None,None,NaN,None,None
19,3021,NaN,None,None,None,NaN,None,None
24,3433,Suspect related to N-Palmitoyl-D-sphingosine (...,N-Palmitoyl-D-sphingosine,C34H65NO3,C34H65NO3,-2.016,2-amino-3-oxo-butanoic_acid|Intact disulfide b...,M-H2O+H
30,2747,Suspect related to Spectral Match to 1-(9Z-Oct...,1-(9Z-Octadecenoyl)-sn-glycero-3-phosphocholine,C24H49N4O6P,C26H44N6O5,-1.011,unspecified,M+Na


### 1.6 Build Feature Base from `b3_iimn_fbmn_quant.csv`

In [10]:
ms2_df = pd.read_csv(f"{DATA_PATH}/b3_iimn_fbmn_quant.csv")
feature_base = ms2_df.rename(columns={'row ID': 'mappingFeatureId', 'row m/z': 'mz', 'row retention time': 'rt'})
print(f"Feature base shape: {feature_base.shape}")
feature_base[['mappingFeatureId', 'mz', 'rt']].head()


Feature base shape: (1482, 20)


,mappingFeatureId,mz,rt
0,2,173.983576,0.531517
1,8,413.211456,0.606817
2,13,232.998187,0.610475
3,17,300.972931,0.611681
4,48,247.013757,0.618486


### 1.7 Merge All Annotations

In [11]:
df_combined = (
    feature_base
    .merge(sirius_annotations, on='mappingFeatureId', how='left')
    .merge(gnps_annotations,   on='mappingFeatureId', how='left')
    .merge(suspect_final,      on='mappingFeatureId', how='left')
)
print(f"Combined df shape: {df_combined.shape}")
print(f"  Total features:             {len(df_combined)}")
print(f"  With GNPS annotations:      {df_combined['gnps_Compound_Name'].notna().sum()}")
print(f"  With SIRIUS structure:      {df_combined['sirius_InChIkey2D'].notna().sum()}")
print(f"  With SIRIUS CANOPUS:        {df_combined['sirius_molecularFormula'].notna().sum()}")
print(f"  With Suspect annotations:   {df_combined['gnps_suspect_compound_name'].notna().sum()}")
df_combined.head()


Combined df shape: (1482, 63)
  Total features:             1482
  With GNPS annotations:      26
  With SIRIUS structure:      694
  With SIRIUS CANOPUS:        694
  With Suspect annotations:   49


,mappingFeatureId,mz,rt,row ion mobility,row ion mobility unit,row CCS,correlation group ID,annotation network number,best ion,auto MS2 verify,...,gnps_NPC#superclass,gnps_NPC#class,gnps_NPC#pathway,gnps_name,gnps_suspect_compound_name,gnps_suspect_sirius_formula,gnps_suspect_buddy_formula,gnps_suspect_delta_mz,gnps_suspect_explanation,gnps_suspect_adduct
0,2,173.983576,0.531517,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8,413.211456,0.606817,NaN,NaN,NaN,16.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,13,232.998187,0.610475,NaN,NaN,NaN,16.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,17,300.972931,0.611681,NaN,NaN,NaN,16.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,48,247.013757,0.618486,NaN,NaN,NaN,16.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 1.8 Assign Annotation Source

In [12]:
has_gnps    = df_combined['gnps_Compound_Name'].notna()
has_sirius  = (
    df_combined['sirius_InChIkey2D'].notna() &
    (df_combined['sirius_high_confidence_sirius'] == True)
)
has_suspect = df_combined['gnps_suspect_compound_name'].notna()

df_combined['annotation_source'] = 'Unannotated'
for idx in df_combined.index:
    sources = []
    if has_gnps.loc[idx]:    sources.append('gnps')
    if has_sirius.loc[idx]:  sources.append('sirius')
    if has_suspect.loc[idx]: sources.append('suspect')
    if sources:
        df_combined.loc[idx, 'annotation_source'] = ':'.join(sources)

both_with_inchikeys = (
    df_combined['gnps_InChIkey2D'].notna() &
    df_combined['sirius_InChIkey2D'].notna()
)
df_combined['inchikey_match'] = None
df_combined.loc[both_with_inchikeys, 'inchikey_match'] = (
    df_combined.loc[both_with_inchikeys, 'gnps_InChIkey2D'] ==
    df_combined.loc[both_with_inchikeys, 'sirius_InChIkey2D']
)
print('=== Annotation Source Counts ===')
print(df_combined['annotation_source'].value_counts())
print(f"\nInChIKey matches (GNPS == SIRIUS): {df_combined['inchikey_match'].sum()}")


=== Annotation Source Counts ===
annotation_source
Unannotated       1327
sirius              80
suspect             41
gnps                21
sirius:suspect       8
gnps:sirius          5
Name: count, dtype: int64

InChIKey matches (GNPS == SIRIUS): 3


### 1.9 Save MS2 Annotations

In [13]:
final_cols_desired = [
    'mappingFeatureId', 'mz', 'rt',
    'correlation group ID', 'best ion', 'auto MS2 verify',
    'identified by n=', 'partners', 'neutral M mass',
    'gnps_Compound_Name', 'gnps_smiles', 'gnps_InChI', 'gnps_InChIkey2D',
    'gnps_ClassyFire#superclass', 'gnps_ClassyFire#class', 'gnps_ClassyFire#subclass',
    'gnps_NPC#superclass', 'gnps_NPC#class', 'gnps_NPC#pathway',
    'sirius_molecularFormula', 'sirius_adduct',
    'sirius_NPC#pathway', 'sirius_NPC#pathway Probability',
    'sirius_NPC#superclass', 'sirius_NPC#superclass Probability',
    'sirius_NPC#class', 'sirius_NPC#class Probability',
    'sirius_ClassyFire#superclass', 'sirius_ClassyFire#superclass probability',
    'sirius_ClassyFire#class', 'sirius_ClassyFire#class Probability',
    'sirius_ClassyFire#subclass', 'sirius_ClassyFire#subclass Probability',
    'sirius_ClassyFire#level 5', 'sirius_ClassyFire#level 5 Probability',
    'sirius_ClassyFire#most specific class', 'sirius_ClassyFire#most specific class Probability',
    'sirius_ConfidenceScoreApproximate', 'sirius_name', 'sirius_InChIkey2D',
    'sirius_InChI', 'sirius_smiles', 'sirius_dbflags', 'sirius_links',
    'sirius_high_confidence_sirius',
    'gnps_suspect_compound_name', 'gnps_suspect_sirius_formula',
    'gnps_suspect_buddy_formula', 'gnps_suspect_delta_mz',
    'gnps_suspect_explanation', 'gnps_suspect_adduct',
    'annotation_source', 'inchikey_match'
]
final_cols = [c for c in final_cols_desired if c in df_combined.columns]
final_ms2_annotations = df_combined[final_cols].copy()
output_path = f"../results/{BATCH_ID}_ms2_annotations.csv"
final_ms2_annotations.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Total features:  {len(final_ms2_annotations)}")
print(f"Annotated:       {(final_ms2_annotations['annotation_source'] != 'Unannotated').sum()}")
print(f"Unannotated:     {(final_ms2_annotations['annotation_source'] == 'Unannotated').sum()}")
final_ms2_annotations.head()


Saved: ../results/b3_cassavaonly_ms2_annotations.csv
Total features:  1482
Annotated:       155
Unannotated:     1327


,mappingFeatureId,mz,rt,correlation group ID,best ion,auto MS2 verify,identified by n=,partners,neutral M mass,gnps_Compound_Name,...,sirius_links,sirius_high_confidence_sirius,gnps_suspect_compound_name,gnps_suspect_sirius_formula,gnps_suspect_buddy_formula,gnps_suspect_delta_mz,gnps_suspect_explanation,gnps_suspect_adduct,annotation_source,inchikey_match
0,2,173.983576,0.531517,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unannotated,None
1,8,413.211456,0.606817,16.0,NaN,NaN,NaN,NaN,NaN,NaN,...,CHEBI:(162963);PUBCHEM:(17907398 145455395),False,NaN,NaN,NaN,NaN,NaN,NaN,Unannotated,None
2,13,232.998187,0.610475,16.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NORMAN:(NS00033255);PUBCHEMANNOTATIONSAFETYAND...,False,NaN,NaN,NaN,NaN,NaN,NaN,Unannotated,None
3,17,300.972931,0.611681,16.0,NaN,NaN,NaN,NaN,NaN,NaN,...,PUBCHEM:(153523186),False,NaN,NaN,NaN,NaN,NaN,NaN,Unannotated,None
4,48,247.013757,0.618486,16.0,NaN,NaN,NaN,NaN,NaN,NaN,...,PUBCHEM:(5866391 3580242),False,NaN,NaN,NaN,NaN,NaN,NaN,Unannotated,None


---
## Section 2 — Quantitative Analysis (CSF only)

Peak areas are normalized by sample weight (1.50 g control, 2.06 g fermented), filtered for prevalence (detected in ≥ 2 of 3 replicates in at least one group), and missing values are imputed with uniform random noise below the second-smallest observed intensity. After log2 transformation, Welch t-tests compare control vs. fermented, with Benjamini-Hochberg FDR correction. Raw p-values are also saved separately for mummichog pathway analysis.

In [14]:
quant_data = pd.read_csv(f"{DATA_PATH}/b3_full_feature_table_cassava.csv")
print(f"Feature table shape: {quant_data.shape}")
quant_data.head()


Feature table shape: (3466, 154)


,id,mz,mz_range:min,mz_range:max,rt,rt_range:min,rt_range:max,area,height,intensity_range:min,...,datafile:Mots_18102024_Foodomics_CSF2_S51.mzML:area,datafile:Mots_18102024_Foodomics_CSF2_S51.mzML:height,datafile:Mots_18102024_Foodomics_CSF2_S51.mzML:intensity_range:min,datafile:Mots_18102024_Foodomics_CSF2_S51.mzML:intensity_range:max,datafile:Mots_18102024_Foodomics_CSF2_S51.mzML:charge,datafile:Mots_18102024_Foodomics_CSF2_S51.mzML:fragment_scans,datafile:Mots_18102024_Foodomics_CSF2_S51.mzML:isotopes,datafile:Mots_18102024_Foodomics_CSF2_S51.mzML:tailing_factor,datafile:Mots_18102024_Foodomics_CSF2_S51.mzML:asymmetry_factor,datafile:Mots_18102024_Foodomics_CSF2_S51.mzML:rt_ms2_apex_distance
0,1,203.05160,203.03696,203.05240,0.5212,0.5013,0.5644,124.50,2911.0,1108.0,...,29.87,2423.0,1108.0,2423.0,NaN,NaN,NaN,0.8395,0.6790,NaN
1,2,173.98358,173.98279,173.98428,0.5315,0.5012,0.5889,276.40,4747.0,1107.0,...,276.40,4747.0,1797.0,4747.0,NaN,NaN,NaN,1.0020,1.0039,NaN
2,3,164.98293,164.98218,164.98354,0.5326,0.5002,0.5801,235.80,4074.0,1273.0,...,104.10,4074.0,2761.0,4074.0,NaN,NaN,NaN,0.6450,0.2899,NaN
3,4,132.95724,132.95659,132.95784,0.5347,0.5002,0.5871,271.30,4747.0,1055.0,...,246.30,4231.0,1809.0,4231.0,NaN,NaN,NaN,1.2669,1.5338,NaN
4,5,308.21663,308.21520,308.21895,0.5596,0.5416,0.5871,73.26,2729.0,828.8,...,17.21,1223.0,1060.0,1223.0,NaN,NaN,NaN,1.6411,2.2821,NaN


In [15]:
area_mapping = {
    'datafile:Mots_18102024_Foodomics_CSF1_S46.mzML:area': 'CSF_1_rep_1',
    'datafile:Mots_18102024_Foodomics_CSF1_S47.mzML:area': 'CSF_1_rep_2',
    'datafile:Mots_18102024_Foodomics_CSF1_S48.mzML:area': 'CSF_1_rep_3',
    'datafile:Mots_18102024_Foodomics_CSF2_S49.mzML:area': 'CSF_2_rep_1',
    'datafile:Mots_18102024_Foodomics_CSF2_S50.mzML:area': 'CSF_2_rep_2',
    'datafile:Mots_18102024_Foodomics_CSF2_S51.mzML:area': 'CSF_2_rep_3',
}
feature_list_renamed = quant_data.rename(columns=area_mapping)
feature_list_clean = feature_list_renamed[['id', 'mz', 'rt'] + list(area_mapping.values())].copy()
print(f"Feature list shape: {feature_list_clean.shape}")
feature_list_clean.head()


Feature list shape: (3466, 9)


,id,mz,rt,CSF_1_rep_1,CSF_1_rep_2,CSF_1_rep_3,CSF_2_rep_1,CSF_2_rep_2,CSF_2_rep_3
0,1,203.05160,0.5212,13.13,8.017,61.79,124.500,38.210,29.87
1,2,173.98358,0.5315,100.70,139.900,13.46,267.000,60.680,276.40
2,3,164.98293,0.5326,235.80,218.100,220.90,162.500,68.360,104.10
3,4,132.95724,0.5347,271.30,58.020,262.90,30.770,25.410,246.30
4,5,308.21663,0.5596,66.56,62.260,73.26,3.793,5.118,17.21


In [16]:
SAMPLE_WEIGHTS = {
    'CSF_1_rep_1': 1.50,  # Cassava Control
    'CSF_1_rep_2': 1.50,
    'CSF_1_rep_3': 1.50,
    'CSF_2_rep_1': 2.06,  # Cassava Fermented
    'CSF_2_rep_2': 2.06,
    'CSF_2_rep_3': 2.06,
}
for sample, weight in SAMPLE_WEIGHTS.items():
    feature_list_clean[f"{sample}_norm"] = feature_list_clean[sample] / weight
print('Normalization complete.')


Normalization complete.


In [17]:
CSF1_cols = [c for c in feature_list_clean.columns if c.startswith('CSF_1') and c.endswith('_norm')]
CSF2_cols = [c for c in feature_list_clean.columns if c.startswith('CSF_2') and c.endswith('_norm')]
all_norm_cols = CSF1_cols + CSF2_cols

def global_prevalence_filter(df, min_present=2):
    csf1_ok = df[CSF1_cols].notna().sum(axis=1) >= min_present
    csf2_ok = df[CSF2_cols].notna().sum(axis=1) >= min_present
    return df[csf1_ok | csf2_ok].copy()

n_before = len(feature_list_clean)
feature_list_filtered = global_prevalence_filter(feature_list_clean)
print(f"Prevalence filter: {n_before:,} → {len(feature_list_filtered):,} features")


Prevalence filter: 3,466 → 3,324 features


In [18]:
np.random.seed(RANDOM_SEED)
feature_list_imputed = feature_list_filtered.copy()

flat_vals     = feature_list_filtered[all_norm_cols].to_numpy(dtype=float).ravel()
positive_vals = flat_vals[np.isfinite(flat_vals) & (flat_vals > 0)]
unique_pos    = np.unique(positive_vals)
second_min    = unique_pos[1] if unique_pos.size > 1 else unique_pos[0]

n_imputed = 0
for col in all_norm_cols:
    mask = feature_list_imputed[col].isna() | (feature_list_imputed[col] == 0)
    if mask.sum():
        noise = np.clip(np.random.uniform(0, second_min, size=mask.sum()), 1e-10, None)
        feature_list_imputed.loc[mask, col] = noise
        n_imputed += mask.sum()

for col in all_norm_cols:
    feature_list_imputed[col.replace('_norm', '_norm_log2')] = np.log2(feature_list_imputed[col])

print(f"Imputed: {n_imputed:,} values | second_min: {second_min:.3e}")
print('Log2 transform complete.')


Imputed: 962 values | second_min: 8.791e-01
Log2 transform complete.


In [19]:
def run_comparison(df, group1_name, group2_name, output_filename, fdr=False):
    g1 = [c for c in df.columns if c.startswith(group1_name) and c.endswith('_norm_log2')]
    g2 = [c for c in df.columns if c.startswith(group2_name) and c.endswith('_norm_log2')]
    log2fc = df[g2].mean(axis=1).values - df[g1].mean(axis=1).values
    t_stat, p_val = stats.ttest_ind(df[g1], df[g2], axis=1, equal_var=False)
    res = df[['id', 'mz', 'rt']].copy()
    res['log2FC'] = log2fc
    res['t.score'] = t_stat
    res['p.value'] = p_val
    res = res.dropna(subset=['t.score', 'p.value'])
    if fdr:
        _, p_adj, _, _ = multipletests(res['p.value'], alpha=0.05, method='fdr_bh')
        res['p.value'] = p_adj
    res = res.sort_values('p.value')
    res.to_csv(output_filename, index=False)
    label = f"{group1_name} vs {group2_name}{'  [FDR-BH]' if fdr else ''}"
    print(f"  {label}")
    print(f"    Tested: {len(res):,} | p<0.05: {(res['p.value']<0.05).sum():,} | |FC|>1 & p<0.05: {((abs(res['log2FC'])>1)&(res['p.value']<0.05)).sum():,}")
    return res

print('='*60)
print(f'CASSAVA — {len(feature_list_imputed):,} features')
print('='*60)
cassava_raw = run_comparison(
    feature_list_imputed, 'CSF_1', 'CSF_2',
    '../results/mummichog_cassava_vs_fermented_cassava.csv', fdr=False
)
cassava_fdr = run_comparison(
    feature_list_imputed, 'CSF_1', 'CSF_2',
    f'../results/{BATCH_ID}_fdr_cassava_vs_fermented_cassava.csv', fdr=True
)
feature_list_imputed.to_csv(f'../results/{BATCH_ID}_feature_list_imputed_log2.csv', index=False)
print(f'\nImputed feature table saved.')


CASSAVA — 3,324 features
  CSF_1 vs CSF_2
    Tested: 3,324 | p<0.05: 1,493 | |FC|>1 & p<0.05: 1,033
  CSF_1 vs CSF_2  [FDR-BH]
    Tested: 3,324 | p<0.05: 1,135 | |FC|>1 & p<0.05: 773

Imputed feature table saved.


---
## Section 3 — BioTransformer Prep (InChIKey → SMILES)

InChI strings from high-confidence SIRIUS and GNPS annotations are converted to canonical SMILES using RDKit, then standardized (fragment parent selection, charge neutralization, tautomer canonicalization). The output CSV feeds into `biotransformer_sirius.py` for metabolic transformation prediction with BioTransformer 3.0.

In [20]:
annotations = pd.read_csv(f'../results/{BATCH_ID}_ms2_annotations.csv')
print(f'Total features: {len(annotations)}')
print(annotations['annotation_source'].value_counts())
annotations = annotations[annotations['annotation_source'] != 'Unannotated'].copy()
gnps_only_mask = annotations['annotation_source'] == 'gnps'
annotations.loc[gnps_only_mask, 'sirius_InChIkey2D'] = np.nan
annotations.loc[gnps_only_mask, 'sirius_name']       = np.nan
print(f'\nAnnotated features for BioTransformer: {len(annotations)}')


Total features: 1482
annotation_source
Unannotated       1327
sirius              80
suspect             41
gnps                21
sirius:suspect       8
gnps:sirius          5
Name: count, dtype: int64

Annotated features for BioTransformer: 155


In [21]:
annotations_with_inchi = annotations[
    annotations['gnps_InChIkey2D'].notna() | annotations['sirius_InChIkey2D'].notna()
][['mappingFeatureId', 'gnps_InChIkey2D', 'sirius_InChIkey2D',
   'gnps_InChI', 'sirius_InChI', 'annotation_source']].copy()

inchikey_list = []
for _, row in annotations_with_inchi.iterrows():
    if pd.notna(row['gnps_InChIkey2D']) and pd.notna(row['gnps_InChI']):
        inchikey_list.append({'inchikey': row['gnps_InChIkey2D'], 'inchi': row['gnps_InChI']})
    if pd.notna(row['sirius_InChIkey2D']) and pd.notna(row['sirius_InChI']):
        inchikey_list.append({'inchikey': row['sirius_InChIkey2D'], 'inchi': row['sirius_InChI']})

only_inchis = pd.DataFrame(inchikey_list).drop_duplicates(subset=['inchikey']).reset_index(drop=True)
print(f'Unique InChIKeys: {len(only_inchis)}')


Unique InChIKeys: 123


In [22]:
from rdkit import Chem
from rdkit.Chem import AllChem

def inchi_to_smiles(inchi):
    try:
        mol = Chem.MolFromInchi(inchi)
        return Chem.MolToSmiles(mol, isomericSmiles=True, canonical=True) if mol else None
    except:
        return None

def clean_and_convert(inchi):
    if pd.isna(inchi): return None
    inchi = str(inchi).strip().strip('"')
    if len(inchi) == 27 and '-' in inchi: return None
    if not inchi.startswith('InChI='): inchi = 'InChI=' + inchi
    return inchi_to_smiles(inchi)

only_inchis['smiles'] = only_inchis['inchi'].apply(inchi_to_smiles)
failed = only_inchis['smiles'].isna()
if failed.any():
    only_inchis.loc[failed, 'smiles'] = only_inchis.loc[failed, 'inchi'].apply(clean_and_convert)
print(f'Total: {len(only_inchis)} | Success: {only_inchis["smiles"].notna().sum()} | Failed: {only_inchis["smiles"].isna().sum()}')


Total: 123 | Success: 123 | Failed: 0


[16:14:57] ERROR: 



In [23]:
from rdkit.Chem.MolStandardize import rdMolStandardize

def standardize(smiles):
    if pd.isna(smiles): return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return smiles
        mol = rdMolStandardize.Cleanup(mol)
        mol = rdMolStandardize.FragmentParent(mol)
        mol = rdMolStandardize.Uncharger().uncharge(mol)
        mol = rdMolStandardize.TautomerEnumerator().Canonicalize(mol)
        return Chem.MolToSmiles(mol, canonical=True)
    except:
        return smiles

only_inchis['clean_smiles'] = only_inchis['smiles'].apply(standardize)
print(f'Standardized: {only_inchis["clean_smiles"].notna().sum()}/{len(only_inchis)}')


[16:14:57] Initializing MetalDisconnector
[16:14:57] Running MetalDisconnector
[16:14:57] Initializing Normalizer
[16:14:57] Running Normalizer
[16:14:57] Initializing MetalDisconnector
[16:14:57] Running MetalDisconnector
[16:14:57] Initializing Normalizer
[16:14:57] Running Normalizer
[16:14:57] Running LargestFragmentChooser
[16:14:57] Running Uncharger
[16:14:57] Initializing MetalDisconnector
[16:14:57] Running MetalDisconnector
[16:14:57] Initializing Normalizer
[16:14:57] Running Normalizer
[16:14:57] Initializing MetalDisconnector
[16:14:57] Running MetalDisconnector
[16:14:57] Initializing Normalizer
[16:14:57] Running Normalizer
[16:14:57] Running LargestFragmentChooser
[16:14:57] Running Uncharger
[16:14:57] Initializing MetalDisconnector
[16:14:57] Running MetalDisconnector
[16:14:57] Initializing Normalizer
[16:14:57] Running Normalizer
[16:14:57] Initializing MetalDisconnector
[16:14:57] Running MetalDisconnector
[16:14:57] Initializing Normalizer
[16:14:57] Running Norma

Standardized: 123/123


[16:14:58] Initializing MetalDisconnector
[16:14:58] Running MetalDisconnector
[16:14:58] Initializing Normalizer
[16:14:58] Running Normalizer
[16:14:58] Initializing MetalDisconnector
[16:14:58] Running MetalDisconnector
[16:14:58] Initializing Normalizer
[16:14:58] Running Normalizer
[16:14:58] Running LargestFragmentChooser
[16:14:58] Running Uncharger
[16:14:58] Initializing MetalDisconnector
[16:14:58] Running MetalDisconnector
[16:14:58] Initializing Normalizer
[16:14:58] Running Normalizer
[16:14:58] Initializing MetalDisconnector
[16:14:58] Running MetalDisconnector
[16:14:58] Initializing Normalizer
[16:14:58] Running Normalizer
[16:14:58] Running LargestFragmentChooser
[16:14:58] Running Uncharger
[16:14:58] Initializing MetalDisconnector
[16:14:58] Running MetalDisconnector
[16:14:58] Initializing Normalizer
[16:14:58] Running Normalizer
[16:14:58] Initializing MetalDisconnector
[16:14:58] Running MetalDisconnector
[16:14:58] Initializing Normalizer
[16:14:58] Running Norma

In [24]:
gnps_df = annotations[['mappingFeatureId', 'annotation_source', 'mz', 'rt',
                        'gnps_InChIkey2D', 'gnps_Compound_Name']].rename(
    columns={'gnps_InChIkey2D': 'inchikey', 'gnps_Compound_Name': 'compound_name'})
sirius_df = annotations[['mappingFeatureId', 'annotation_source', 'mz', 'rt',
                          'sirius_InChIkey2D', 'sirius_name']].rename(
    columns={'sirius_InChIkey2D': 'inchikey', 'sirius_name': 'compound_name'})
combined_annotations = pd.concat([gnps_df, sirius_df]).drop_duplicates()
result = only_inchis.merge(combined_annotations, on='inchikey', how='left')
result = result[['mappingFeatureId', 'inchikey', 'inchi', 'smiles', 'clean_smiles',
                 'annotation_source', 'mz', 'rt', 'compound_name']]
print(f'BioTransformer input shape: {result.shape}')
result.head()


BioTransformer input shape: (140, 9)


,mappingFeatureId,inchikey,inchi,smiles,clean_smiles,annotation_source,mz,rt,compound_name
0,287,GFFGJBXGBJISGV,InChI=1S/C5H5N5/c6-4-3-5(9-1-7-3)10-2-8-4/h1-2...,Nc1nc[nH]c2ncnc1-2,Nc1ncnc2[nH]cnc12,sirius,136.060796,0.855350,Adenin
1,315,UYTPUPDQBNUYGX,InChI=1S/C5H5N5O/c6-5-9-3-2(4(11)10-5)7-1-8-3/...,N=c1nc(O)c2nc[nH]c2[nH]1,Nc1nc(=O)c2[nH]cnc2[nH]1,sirius,152.055677,0.877261,Guanine
2,325,OANCGDPJMNPBDV,InChI=1S/C11H21NO7/c1-5(2)7(10(16)17)12-4-11(1...,CC(C)C(NCC1(O)OC(CO)C(O)C1O)C(=O)O,CC(C)C(NCC1(O)OC(CO)C(O)C1O)C(=O)O,sirius,280.136972,0.902022,"3-methyl-2-({[2,3,4-trihydroxy-5-(hydroxymethy..."
3,347,FDGQSTZJBFJUBT,InChI=1S/C5H4N4O/c10-5-3-4(7-1-6-3)8-2-9-5/h1-...,Oc1ncnc2[nH]cnc12,O=c1[nH]cnc2[nH]cnc12,sirius,137.044836,0.951722,Sarkin
4,352,OUYCCCASQSFEME,InChI=1S/C9H11NO3/c10-8(9(12)13)5-6-1-3-7(11)4...,NC(Cc1ccc(O)cc1)C(=O)O,NC(Cc1ccc(O)cc1)C(=O)O,sirius,182.080128,0.962800,L-Tyr


In [ ]:
only_inchis.to_csv(f'../results/{BATCH_ID}_clean_inchis_smiles.csv', index=False)
print(f'Saved: ../results/{BATCH_ID}_clean_inchis_smiles.csv')

bt_dir = '../data/biotransformer'
os.makedirs(bt_dir, exist_ok=True)
output_file = os.path.join(bt_dir, f'{BATCH_ID}_compounds_for_biotransformer.csv')
result.to_csv(output_file, index=False)
print(f'Saved: {output_file}')
print(f'  Features: {len(result)}')
print(f'  Columns:  {list(result.columns)}')
print('\n✓ Ready to run: python scripts/biotransformer_sirius.py --batch b3_cassavaonly')
print('\n If _biot files were run then please ignore the aforementioned step and just continue with figure generation')


Saved: ../results/b3_cassavaonly_clean_inchis_smiles.csv
Saved: ../data/biotransformer/b3_cassavaonly_compounds_for_biotransformer.csv
  Features: 140
  Columns:  ['mappingFeatureId', 'inchikey', 'inchi', 'smiles', 'clean_smiles', 'annotation_source', 'mz', 'rt', 'compound_name']

✓ Ready to run: python scripts/biotransformer_sirius.py --batch b3_cassavaonly
